In [1]:
from PIL import Image, ImageEnhance
import math
import os

def img_watermark(image_name, image_path):
    # 경로 설정
    parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
    output_dir = os.path.join(parent_dir, 'wm_uploads')
    os.makedirs(output_dir, exist_ok=True)

    # 1. 원본 이미지 처리
    original = Image.open(image_path)
    file_ext = os.path.splitext(image_name)[1].lower()
    
    # JPG 대응: RGB 모드로 변환
    if original.mode != 'RGBA':
        image = original.convert('RGBA')
    else:
        image = original.copy()

    # 2. 워터마크 로고 준비 (한 번만 로드)
    logo = Image.open("logo.png").convert("RGBA")
    alpha = logo.split()[3]
    alpha = ImageEnhance.Brightness(alpha).enhance(0.6)
    logo.putalpha(alpha)
    logo_width, logo_height = logo.size

    # 3. 워터마크 배치 계산
    width, height = image.size
    interval_x = math.trunc(width / 35)*10 if width > 600 else 200
    interval_y = 200 if height > 600 else math.trunc(height / 30)*10

    padding = 15
    usable_height = height - 2 * padding - logo_height
    num_lines = max(2, int(usable_height // interval_y) + 1)

    # 4. 워터마크 레이어 생성
    watermark_layer = Image.new("RGBA", (width, height), (0, 0, 0, 0))
    if num_lines == 2:
        y_coords = [padding, height - padding - logo_height]
    else:
        step = usable_height / (num_lines - 1)
        y_coords = [int(padding + i * step) for i in range(num_lines)]

    for y in y_coords:
        for x in range(0, width + interval_x, interval_x):
            watermark_layer.paste(logo, (x, y), logo)

    # 5. 회전 처리 (크기 유지)
    rotated_watermark = watermark_layer.rotate(
        45, 
        expand=False,  # 크기 변경 없음
        center=(width//2, height//2)
    )

    # 6. 이미지 합성
    watermarked = Image.alpha_composite(image, rotated_watermark)

    # 7. 저장 모드 결정
    save_path = os.path.join(output_dir, f"wm_{image_name}")
    
    if file_ext in ('.jpg', '.jpeg'):
        watermarked = watermarked.convert('RGB')  # 알파 채널 제거
        watermarked.save(save_path, quality=95, optimize=True)
    else:
        watermarked.save(save_path)

    return save_path


In [2]:
# !pip install PyMuPDF

In [3]:
import os
import fitz  # PyMuPDF 임포트
def pdf_watermark(pdf_name, path):
    
    # 상위 폴더 경로 계산
    parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
    output_dir = os.path.join(parent_dir, 'wm_uploads')
    
    # 출력 폴더 생성 (없을 경우)
    os.makedirs(output_dir, exist_ok=True)
    
    # 파일 경로 설정
    original_file = path
    watermark_file = 'WaterMark.pdf'
    new_file = os.path.join(output_dir, f"wm_{pdf_name}")
    
    # PDF 워터마킹 처리
    original_pdf = fitz.open(original_file)
    watermark_pdf = fitz.open(watermark_file)
    
    for page_num in range(len(original_pdf)):
        page = original_pdf[page_num]
        page.show_pdf_page(page.rect, watermark_pdf, 0)
    
    original_pdf.save(new_file)
    return new_file  # 전체 저장 경로 반환

In [4]:
# !pip install flask

In [ ]:
from flask import Flask, request
from flask import Response
from flask import jsonify
from concurrent.futures import ThreadPoolExecutor

app = Flask(__name__)
app.config['JSON_AS_ASCII'] = False


@app.route('/watermark', methods=['GET'])
def watermark():
    try:
        files = request.get_json()['files']
        print("========================================================")
        print("받은 files:", files)
        if not files or not isinstance(files, list):
            return jsonify({"error": "Invalid file list"}), 400

        file_info = [
            (file['filename'], file['path'], file['path'].split(".")[-1].lower())
            for file in files
        ]

        print("========================================================")
        print("정제한 files:", file_info)

        # 쓰레드를 통해 다중 처리
        processed_paths = []
        with ThreadPoolExecutor() as executor:
            futures = []
            for filename, path, ext in file_info:
                if ext == 'pdf':
                    futures.append(executor.submit(pdf_watermark, filename, path))
                else:
                    futures.append(executor.submit(img_watermark, filename, path))
            
            for future in futures:
                result = future.result()
                processed_paths.append(result)

        return jsonify({"wm_path": processed_paths}), 200

    except Exception as e:
        print(e)
        return jsonify({"error": str(e)}), 500



# 코드수정시 자동반영
if __name__ == "__main__":
    app.run()

 * Serving Flask app '__main__'


 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [16/Jun/2025 15:20:15] "GET /watermark HTTP/1.1" 200 -


받은 files: [{'fieldname': 'files', 'originalname': 'poster.jpg', 'encoding': '7bit', 'mimetype': 'image/jpeg', 'encodingName': 'poster.jpg', 'destination': 'C:\\Users\\user6\\project\\IdeaLink\\uploads', 'filename': 'poster-1750054815183-356965669.jpg', 'path': 'C:\\Users\\user6\\project\\IdeaLink\\uploads\\poster-1750054815183-356965669.jpg', 'size': 258321}]
정제한 files: [('poster-1750054815183-356965669.jpg', 'C:\\Users\\user6\\project\\IdeaLink\\uploads\\poster-1750054815183-356965669.jpg', 'jpg')]


127.0.0.1 - - [16/Jun/2025 15:23:28] "GET /watermark HTTP/1.1" 200 -


...

받은 files: [{'fieldname': 'files', 'originalname': 'poster.jpg', 'encoding': '7bit', 'mimetype': 'image/jpeg', 'encodingName': 'poster.jpg', 'destination': 'C:\\Users\\user6\\project\\IdeaLink\\uploads', 'filename': 'poster-1750059825473-116086469.jpg', 'path': 'C:\\Users\\user6\\project\\IdeaLink\\uploads\\poster-1750059825473-116086469.jpg', 'size': 258321}]
정제한 files: [('poster-1750059825473-116086469.jpg', 'C:\\Users\\user6\\project\\IdeaLink\\uploads\\poster-1750059825473-116086469.jpg', 'jpg')]


127.0.0.1 - - [17/Jun/2025 12:40:04] "GET /watermark HTTP/1.1" 200 -


받은 files: [{'fieldname': 'files', 'originalname': 'cat.1008.jpg', 'encoding': '7bit', 'mimetype': 'image/jpeg', 'encodingName': 'cat.1008.jpg', 'destination': 'C:\\Users\\user6\\project\\IdeaLink\\uploads', 'filename': 'cat.1008-1750131604684-604005791.jpg', 'path': 'C:\\Users\\user6\\project\\IdeaLink\\uploads\\cat.1008-1750131604684-604005791.jpg', 'size': 25939}]
정제한 files: [('cat.1008-1750131604684-604005791.jpg', 'C:\\Users\\user6\\project\\IdeaLink\\uploads\\cat.1008-1750131604684-604005791.jpg', 'jpg')]


127.0.0.1 - - [17/Jun/2025 12:40:29] "GET /watermark HTTP/1.1" 200 -


받은 files: [{'fieldname': 'files', 'originalname': 'ocr.jpg', 'encoding': '7bit', 'mimetype': 'image/jpeg', 'encodingName': 'ocr.jpg', 'destination': 'C:\\Users\\user6\\project\\IdeaLink\\uploads', 'filename': 'ocr-1750131629644-322045077.jpg', 'path': 'C:\\Users\\user6\\project\\IdeaLink\\uploads\\ocr-1750131629644-322045077.jpg', 'size': 264762}]
정제한 files: [('ocr-1750131629644-322045077.jpg', 'C:\\Users\\user6\\project\\IdeaLink\\uploads\\ocr-1750131629644-322045077.jpg', 'jpg')]
